# Multi-Agent Harness: 압류 검토 · 자금 흐름 · AML 조사 확대

**요청**: 고객 A의 압류명령서, KYC, 거래 원장을 확인하고 자금 흐름을 분석하여 검토 보고서를 작성합니다.

이 Hands-on은 합성 데이터와 모의 OCR 텍스트만 사용합니다. 문서의 적법성, 압류 대상 자산, 의심거래 여부는 LLM·규칙·그래프가 단독으로 결정하지 않으며, 법무·준법감시인의 승인이 필요합니다.

## 실행 설계

```
Coordinator
  ├─ [병렬] Document / KYC / Ledger / Compliance
  ├─ Graph: 원장 결과로 자금 흐름 분석
  ├─ Replan: 계좌 별지 미첨부 → 법무 확인 단계 추가
  ├─ Dynamic delegation: 반복 rapid pass-through → AML Agent
  └─ Report → 법무·준법감시인 승인
```

기본 실행은 외부 API, 실제 PDF/OCR, SAP, 계정·자산 변경을 호출하지 않습니다. `5_harness`의 routing과 Deep Orchestrator 프롬프트를 연결 경계로 사용합니다.

## Gradio Multi-Agent Harness UI

아래 셀을 실행하면 Gradio 화면에서 요청 입력, Agent trace, 근거 요약, 자금 흐름, 보고서 초안, 승인 통제를 한 번에 확인합니다. UI는 합성 fixture만 사용하며 외부 시스템을 호출하지 않습니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(ROOT / 'shared'), str(ROOT / '5_harness'), str(ROOT / '6_Business_issue' / 'src')]
from seizure_aml_multiagent import build_financial_multiagent_ui

financial_multiagent_ui = build_financial_multiagent_ui()
financial_multiagent_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=980)


In [ ]:
# 외부 runtime 없이 병렬 역할·재계획·동적 위임을 작은 함수로 재현합니다.
case = {"request": "압류 문서·KYC·거래 원장을 검토해 보고서를 작성해줘.", "annex_missing": True, "transfers": [{"from": "EXT-01", "to": "DEMO-AC-01", "amount": 5_000_000}, {"from": "DEMO-AC-01", "to": "EXT-02", "amount": 4_800_000}]}
def run_multi_agent(data):
    document = {"agent": "document", "annex_missing": data["annex_missing"]}; kyc = {"agent": "kyc", "screening": "clear_in_snapshot"}; ledger = {"agent": "ledger", "transfers": data["transfers"]}
    first, second = data["transfers"]; graph = {"agent": "graph", "rapid_pass_through_events": [second] if second["amount"] >= first["amount"] * .9 else []}; aml = {"agent": "aml", "assessment": "review_required"} if graph["rapid_pass_through_events"] else None
    trace = [{"agent": x["agent"], "event": "completed"} for x in [document, kyc, ledger, graph]]
    if document["annex_missing"]: trace.append({"agent": "coordinator", "event": "replanned", "reason": "계좌 별지 누락"})
    if aml: trace.append({"agent": "coordinator", "event": "delegated", "delegate": "aml"})
    return {"artifacts": {"document": document, "kyc": kyc, "ledger": ledger, "graph": graph, "aml": aml}, "trace": trace, "action": "legal_and_compliance_approval_required", "asset_restriction_performed": False, "aml_report_filed": False}


## 1. 입력 확인 및 선택 기능 준비 상태

`deepagents`/`langgraph`는 실제 runtime 연결을 위한 선택 의존성이고, `networkx`/`pyvis`는 자금 흐름 시각화 선택 기능입니다. 기본 mock workflow는 설치 없이 실행됩니다.

In [ ]:
print(case["request"]); print("입력은 합성 read-only snapshot입니다.")


## 2. Multi-Agent 실행과 관찰성

첫 단계의 네 Agent는 병렬로 읽기 전용 근거를 수집합니다. Graph 분석 뒤 Coordinator는 결과를 평가해 계획을 수정하고 필요한 경우에만 AML Agent를 추가 호출합니다.

In [ ]:
result = run_multi_agent(case)
for event in result["trace"]: print("-", event)


## 3. Graph Agent와 AML 동적 위임

반복적인 rapid pass-through는 조사 **신호**일 뿐 불법이나 의심거래의 확정이 아닙니다. 이 예제에서는 임계치에 도달했기 때문에 AML Agent를 추가 호출하고 `review_required`로만 표시합니다.

In [ ]:
graph = result["artifacts"]["graph"]
print("Rapid pass-through 후보:", len(graph["rapid_pass_through_events"])); print("AML 결과:", result["artifacts"]["aml"]["assessment"])


## 4. 검토 보고서와 Human-in-the-loop

계좌 별지 미첨부는 Coordinator의 재계획 사유입니다. 추출된 고객 계좌는 원장 증거일 뿐, 압류 대상이라는 결론이 아닙니다. 최종 보고서는 집행·동결·AML 신고를 수행하지 않습니다.

In [ ]:
print("최종 상태:", result["action"]); print("자산 제한 수행:", result["asset_restriction_performed"]); print("AML 보고 제출:", result["aml_report_filed"])


### 실제 연계로 확장할 때

- Document Agent를 승인된 PDF parser/OCR 및 문서보관소의 읽기 전용 도구로 교체합니다.
- KYC·Ledger Agent를 SAP API 또는 MCP read-only tool로 교체하고 최소 권한·감사로그를 적용합니다.
- DeepAgents/LangGraph runtime은 `5_harness/requirements-deepagents.txt`의 선택 의존성 및 연결 권한이 준비된 뒤 연결합니다.
- 자산 제한·외부 신고·계정 변경은 여기와 분리된 승인형 write tool과 직무분리를 거친 뒤에만 구현합니다.